Before we begin, let us execute the below cell to display information about the NVIDIA® CUDA® driver and the GPUs running on the server by running the `nvidia-smi` command. To do this, execute the cell block below by clicking on it with your mouse, and pressing Ctrl+Enter, or pressing the play button in the toolbar above. You should see some output returned below the grey cell.

In [ ]:
nvidia-smi

# Learning objectives
The **goal** of this lab is to:

- Understand what CUDA is and how it supports parallel programming on GPUs.
- Examine the basic terms and steps involved in making a sequential code parallel using CUDA. 
- Apply appropriate techniques for allocating and deallocating memory for CPU and GPU in C/C++.
- Utilise appropriate techniques for moving memory between CPU and GPU using CUDA API in C/C++.
- Select appropriate techniques for GPU functions and how to expose them to be callable from the CPU.
- Describe the CUDA architecture from hardware and software viewpoints.

We do not intend to cover:
- Optimization techniques like memory access patterns and memory hierarchy.

**NOTE**: To be able to see the Nsight Systems profiler output, please download the latest version of Nsight Systems from [here](https://developer.nvidia.com/nsight-systems).

# Introduction
Graphics Processing Units (GPUs) were initially designed to accelerate graphics processing, but in 2007 the release of CUDA introduced GPUs as General Purpose Processors. CUDA is a parallel computing platform and programming model that makes using a GPU for general-purpose computing simple and elegant. The developer still programs in the familiar C, C++, Fortran, or an ever-expanding list of supported languages and incorporates extensions of these languages in the form of a few basic keywords.

- CUDA C/C++ is based on a standard C/C++
- CUDA is a set of extensions to enable heterogeneous programming
- CUDA is a straightforward API to manage devices, memory, etc.

# CUDA
**Heterogeneous Computing:** CUDA is a heterogeneous programming model that includes provisions for a CPU and GPU. The CUDA C/C++ programming interface consists of C language extensions. These enable you to target portions of source code for parallel execution on the device (GPU). CUDA provides a library of C functions that can be executed on the host (CPU) to interact with the device. The two processors that work with each other are:

- Host: CPU and its memory (Host Memory)
- Device: GPU and its memory  (Device Memory)


Let us look at a Hello World example

<summary markdown="span"><b>Example CUDA code</b></summary>

```c++
_global__ void print_from_gpu(void) {
    printf("Hello World! from thread [%d,%d] from device\n", threadIdx.x,blockIdx.x);
}

int main(void) {
    printf("Hello World from host!\n");
    print_from_gpu<<<1,1>>>();
    cudaDeviceSynchronize();
    return 0;
}

```

So you might have already observed that CUDA C is nothing but extensions/constructs to existing language. Let us look at   the additional constructs we introduced above:

- `__global__` :This keyword, when added before the function, tells the compiler that this is a function that will run on the device and not on the host. 
- `<<<,>>>` : This keyword tells the compiler that this is a call to the device function and not the host function. Additionally, the 1,1 parameter dictates the number of threads to launch in the kernel. We will cover the parameters inside the angle brackets later.
- `threadIdx.x, blockIdx.x` : This is a unique ID that's given to all threads.
- `cudaDeviceSynchronize()` : All of the kernel(Function that runs on GPU) calls in CUDA are asynchronous in nature. This API will make sure that the host does not proceed until all device calls are over.

## GPU Architecture
This section will take an approach to describe the CUDA programming model by showing the relationship between the software programming concepts and how they get mapped to GPU hardware.

The diagram below shows a higher level of abstraction of components of GPU hardware and its respective programming model mapping. 

<img src="../../_common/images/cuda_hw_sw.png" width="80%" height="80%">

As shown in the diagram above CUDA programming model is tightly coupled with hardware design. This makes CUDA one of the most efficient parallel programming models for shared memory systems. Another way to look at the diagram shown above is given below: 

| Software | Executes  | Hardware |
| --- | --- | --- |
| CUDA thread  | on/as | CUDA Core | 
| CUDA block  | on/as | Streaming Multiprocessor |
| GRID/Kernel  | on/as | GPU Device |

We will understand the concept of blocks and threads in the upcoming section. But let us first look at the steps involved in writing CUDA code.

## Steps in CUDA Programming

The below table highlights the typical steps which are required to convert sequential code to CUDA code:

| Sequential code | CUDA Code |
| --- | --- |
| **Step 1** Allocate memory ( _malloc, new_ ) | **Step 1**  Allocate memory on Host (_malloc, new_ )|
| **Step 2** Populate/initialize data | **Step 2** Allocate memory on Device, using API like _cudaMalloc()_ |
| **Step 3** Call the function that crunches the data | **Step 3**  Populate/initialize data on Host |
| **Step 4** Consume the crunched data | **Step 4** Transfer the data from Host to Device with _cudaMemcpy()_ |
| **Step 5** Free memory | **Step 5** Call the GPU function with _<<<,>>>_ brackets |
| | **Step 6** Synchronize Device and Host with _cudaDeviceSynchronize()_ |
| | **Step 7** Transfer data from Device to Host with _cudaMemcpy()_ |
| | **Step 8** Consume the crunched data on Host |
| | **Step 9** Free memory on Host |
| | **Step 10** Free memory on Device |

CPU and GPU memory is different, and the developer needs to use additional CUDA API to allocate and free memory on GPU. The only device memory can be consumed inside the GPU function call (kernel).
    
In CUDA C/C++, linear memory on the device is typically allocated using ```cudaMalloc()``` and freed using ```cudaFree()``` and data transfer between host memory and device memory are typically done using ```cudaMemcpy()```.

The API definition of these are as follows: 

**cudaError_t cudaMalloc (void ∗∗ devPtr, size_t size)** allocate size bytes of linear memory on the device and returns a pointer to the allocated memory. The allocated memory is suitably aligned for any kind of variable. `cudaMalloc()` returns ```cudaErrorMemoryAllocation``` in case of failure or ```cudaSuccess```.
    
**cudaError_t cudaMemcpy (void ∗ dst, const void ∗ src, size_t count, enum cudaMemcpyKind kind)** copies count bytes from the memory area pointed to by `src` to the memory area pointed to by `dst`. `dst` and `src` may be any device or host, scalar or array.  `kind` is one of the defined enums `cudaMemcpyHostToDevice`, `cudaMemcpyDeviceToHost`, `cudaMemcpyDeviceToDevice` or `cudaMemcpyHostToHost` (this specifies the direction of the copy).

Please note, calling `cudaMemcpy()` with `dst` and `src` pointers that do not match the direction of the copy results in an undefined behavior.

**cudaError_t cudaFree (void ∗ devPtr)** Frees the memory space pointed to by `devPtr`, which must have been returned by a previous call to `cudaMalloc()` or another equivalent API. 
    
Let us look at these steps in more detail for a simple vector addition code:


<summary markdown="span"><b>Example CUDA code for vector addition</b></summary>
    
```c++
int main(void) {
  // Host copies of a, b, c
  int *h_a, *h_b, *h_c;
  // Device copies of a, b, c
  int *d_a, *d_b, *d_c;
  int threads_per_block = 0, no_of_blocks = 0;

  unsigned n = 1;
  int size = n * sizeof(int);

  // 1 Allocate memory on Host
  h_a = (int *)malloc(size);
  h_b = (int *)malloc(size);
  h_c = (int *)malloc(size);

  // 2 Allocate memory on Device
  cudaMalloc((void **)&d_a, size);
  cudaMalloc((void **)&d_b, size);
  cudaMalloc((void **)&d_c, size);

  // 3 Populate/initialize Host
  fill_array(n, h_a);
  fill_array(n, h_b);

  // 4 Transfer the data from Host to Device with cudaMemcpy()
  cudaMemcpy(d_a, h_a, size, cudaMemcpyHostToDevice);
  cudaMemcpy(d_b, h_b, size, cudaMemcpyHostToDevice);

  // 5 Call the GPU function
  device_add<<<1, 1>>>(d_a, d_b, d_c);

  // 6 Synchronize Device and Host
  cudaDeviceSynchronize();

  // 7 Transfer data from Device to Host with cudaMemcpy()
  cudaMemcpy(h_c, d_c, size, cudaMemcpyDeviceToHost);

  // 8 Consume the crunched data on Host
  for (int idx = 0; idx < n; idx++) {
    printf(" %d + %d  = %d\n", h_a[idx], h_b[idx], h_c[idx]);
  }

  // 9 Free memory on Host
  free(h_a);
  free(h_b);
  free(h_c);
    
  // 10 Free memory on Device
  cudaFree(d_a);
  cudaFree(d_b);
  cudaFree(d_c);

  return 0;
}
```


## Understanding Threads and Blocks
We will be looking at understanding _thread_ and _block_ level parallelism in this section.The number of threads and blocks to be launched is passed as a parameter to ```<<<,>>>``` brackets in a kernel call.

### Creating multiple blocks

In order to create multiple blocks for the vector addition code above, you need to change two things:
1. Change _<<<1, 1>>>_ to <<<N, 1>>>_ which launches N number of blocks
2. Access the array with block index using private variable passed by default to CUDA kernel: _blockIdx.x_

<summary markdown="span"><b>Syntax</b></summary>
    
```c++
//changing from device_add<<<1, 1>>> to
device_add<<<n,1>>>
//access the array using blockIdx.x private variable
__global__ void device_add(int *a, int *b, int *c) {
    c[blockIdx.x] = a[blockIdx.x] + b[blockIdx.x];
}
```

By using `blockIdx.x` to index the array, each block handles a different element of the array and may execute in parallel to each other.

| Block Id | Performs |
| --- | --- |
| Block 0 | _c\[0\]=b\[0\]+a\[0\]_ |
| Block 1 | _c\[1\]=b\[1\]+a\[1\]_ |
| Block 2 | _c\[2\]=b\[2\]+a\[2\]_ |

**Understand and analyze** the sample vector addition code [vector_addition_block.cu](../source_code/vector_addition_gpu_block_only.cu). Open the downloaded files for inspection. 

### Creating multiple threads

In order to create multiple threads for vector addition code above. You need to change two things:
1. change _<<<1, 1>>>_ to <<<1, N>>>_ which launches N number of threads inside 1 block
2. Access the array with thread index using private variable passed by default to CUDA kernel: _threadIdx.x_

<summary markdown="span"><b>Syntax</b></summary>

```c++
//changing from device_add<<<1,1>>> to
device_add<<<1,n>>>
//access the array using threadIdx.x private variable
__global__ void device_add(int *a, int *b, int *c) {
    c[threadIdx.x] = a[threadIdx.x] + b[threadIdx.x];
}
```

By using `threadIdx.x` to index the array, each thread handles a different element of the array and can execute in parallel.

| thread Id | Performs |
| --- | --- |
| Thread 0 | _c\[0\]=b\[0\]+a\[0\]_ |
| Thread 1 | _c\[1\]=b\[1\]+a\[1\]_ |
| Thread 2 | _c\[2\]=b\[2\]+a\[2\]_ |

**Understand and analyze** the sample vector addition code [vector_addition_thread.cu](../source_code/vector_addition_gpu_thread_only.cu).
    
### Creating multiple blocks each having many threads

So far, we've looked at parallel vector addition through the use of several blocks with one thread and one block with several
threads. Now let us look at creating multiple blocks, each block containing multiple threads.

To understand it lets take a scenario where the total number of vector elements is 32 which needs to be added in parallel. Total number of parallel execution unit required is 32. As a first step let us define that each block contains eight threads(we are not saying this is optimal configuration and is just for explanation purpose). Next we define the number of blocks. The simplest calculation is No_Of_Blocks = 32/8 where 8 is number of threads per blocks. The code changes required to launch 4 blocks with 8 thread each is as shown below: 
1. Change _<<<1, 1>>>_ to <<<4, 8>>>_ which launches 8  threads per block and 4 total blocks
2. Access the array with both thread index and block index using private variable passed by default to call CUDA kernel: _threadIdx.x_ and _blockIdx.x_ and _bloxkDim.x_ which tells how many threads are allocated per block. 

    
<summary markdown="span"><b>Syntax</b></summary>

```c++
threads_per_block = 8;
no_of_blocks = n / threads_per_block;
device_add<<<no_of_blocks, threads_per_block>>>(d_a, d_b, d_c);

__global__ void device_add(int *a, int *b, int *c) {
    // Get our global thread ID
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    c[idx] = a[idx] + b[idx];
}
```

The diagram below shows the launch configuration that we have discussed so far:

<img src="../../_common/images/cuda_indexing.png">

Modern GPU Architectures consist of multiple SM (streaming multiprocessors), each consisting of several cores. To utilize the whole GPU, it is important to use both threads and blocks.

**Understand and analyze** the sample vector addition code [vector_addition_block_thread.cu](../source_code/vector_addition_gpu_thread_block.cu).Open the downloaded files for inspection. 

The more important question may arise: why bother with threads altogether? What do we gain by adding an additional level of parallelism? The short answer is CUDA programming model defines that, unlike parallel blocks, threads have mechanisms to efficiently communicate and synchronize.
    
This is necessary to implement certain algorithms where threads needs to communicate with each other. We do not require synchronization across threads in **Pair Calculation** so we will not be going into details of concept of synchronization across threads and usage of specialized memory like _shared_ memory in this tutorial.  

## Unified Memory
An easier way to allocate memory accessible by the GPU is to use *Unified Memory*. It provides a single memory space accessible by all GPUs and CPUs in the system. To allocate data in unified memory, we call `cudaMallocManaged()`, which returns a pointer that you can access from host (CPU) code or device (GPU) code. To free the data, just pass the pointer to `cudaFree()`. To read more about unified memory, please review the blog on [Unified Memory for CUDA beginners](https://developer.nvidia.com/blog/unified-memory-cuda-beginners/).

<img src="../../_common/images/unified_memory.png">

Below is the example usage of how to use managed memory in the CUDA code:

<summary markdown="span"><b>Syntax</b></summary>

```c++
  // Allocate Unified Memory -- accessible from CPU or GPU
  int *a, *b, *c;
  int size = n * sizeof(int);
  cudaMallocManaged(&a, size);
  cudaMallocManaged(&b, size);
  cudaMallocManaged(&c, size);
  ...

  // Free memory
  cudaFree(a);
  cudaFree(b);
  cudaFree(c);
```

**Understand and analyze** the sample vector addition code [vector_addition_block_thread_unified.cu](../source_code/vector_addition_gpu_thread_block_unified.cu).Open the downloaded files for inspection. 

## Atomic functions
In the exercise code you will also require one more operator which will help you get the right results.  CUDA atomic functions ensures that a particular variable operated on by the function is accessed and/or updated atomically to prevent indeterminate results and race conditions. In other words, it prevents one thread from stepping on the toes of other threads due to accessing a variable simultaneously, resulting in different results run-to-run. For example, if we want to accumulate numbers up to N, we could write the following:

<summary markdown="span"><b>Syntax</b></summary>
    
```c++
sum_to<<<1, N>>>
__global__ void sum_to(int *cnt)
{
  atomicAdd(&cnt[0], 1);
}
```

To read more about the available atomic CUDA functions, please review the user documentation on [atomic function](https://docs.nvidia.com/cuda/cuda-c-programming-guide/index.html#atomic-functions)
    
# A Quick Recap
We saw the definition of CUDA and briefly covered CUDA architecture and introduced CUDA C and CUDA Fortran constructs. We also played with block and thread configurations for a simple vector addition code. All this was done under the following restrictions:
1. **Multiple Dimension**: We launched threads and blocks in one dimension. We have been using `threadIdx.x` and `blockIdx.x`, so what is `.x` ? This statement  says that we are launching threads and blocks in one dimension only. CUDA allows to launch threads in 3 dimensions. You can also have `.y` and `.z` for index calculation. For example, you can launch threads and blocks in 2 dimensions to  divide work for a 2D image. Also the maximum number of threads per block and number of blocks allowed per dimension is restricted based on the GPU that the code runs on.
2. **GPU Memory**: What we have not covered is that GPU has different hierarchy of memory, e.g. GPU has a read only memory which provides high bandwidth for 2D and 3D locality access called _texture_. Also, GPU provides a scratch pad with limited memory called  _shared memory_
3. **Optimization** : What we did not cover so far is the right way to access the compute and memory to get max performance. 

**One key characteristic of CUDA is that a user can control the access pattern of data for each thread. The user can decide which part of the memory the data can sit on.  While we are covering some part of this in this lab, which is required for us to port our code, we do not intend to cover all optimizations**

# CUDA Exercise
Now, let's start modifying the original code and add the CUDA constructs. You can either explicitly transfer the allocated data between the CPU and GPU or use unified memory, which creates a pool of managed memory shared between the CPU and GPU.

## Edit the Code

**Click on this <b>[source code](../source_code/rdf.cu)</b> link to start modifying the RDF code.**

To help you modify the code, some sections are marked with `TODO: ` comments consisting of simple instructions. Where additional modifications from the [original source code](../../_common/source_code/rdf.cpp) were required they are marked with `Note: ` comments for you to review. Remember to **SAVE** your code after changes, before running the below cells.

**Note:** When `-arch=native` compiled option is used, `nvcc` detects the visible GPUs on the system and generates codes for them. It willl produce a warning if there is no visible supported GPU on the system, and the default architecture will be used.

## Compile and run the code

In [ ]:
#compile for GPU
cd ../source_code && printf "Compiling CUDA for GPU ...\n" && nvcc -arch=native -o rdf_c rdf.cu &&
printf "\nRunning the executable and validating the output\n" && ./rdf_c && cat Pair_entropy.dat

The output should be the following:

```
s2 value is -2.43191
s2bond value is -3.87015
```

Now, let's profile the code.

In [ ]:
#profile and see output of nvptx
cd ../source_code && nsys profile -t nvtx,cuda --stats=true --force-overwrite true -o rdf_cuda_c ./rdf_c

Let's checkout the profiler's report. Download and save the report file by holding down the Shift key and right-clicking the [report link](../source_code/rdf_cuda_c.nsys-rep) then choosing Save Link As. Once done, open it via the GUI. Have a look at the example expected profiler report below:

**Example screenshot**

<img src="../../_common/images/cuda_profile_timeline_cpp.png" width="80%" height="80%">

Nsight systems is capable of capturing information about CUDA execution in the profiled process. CUDA API row in the _Timeline View_ shows traces of CUDA Runtime and Driver calls made by the application. If you hover your mouse over it, you will see more information about the calls.


Feel free to checkout the [solution with unified (managed) memory](../source_code/SOLUTION/rdf_unified_memory.cu) and the [solution without managed memory](../source_code/SOLUTION/rdf_unmanaged.cu) to help you understand better.


# Analysis

**Usage Scenarios**

Using language  extensions like CUDA C, CUDA Fortran helps developers get the best performance out of their code on an NVIDIA GPU. CUDA C and other language construct exposes the GPU architecture and programming model which gives more control to developers with respect to memory storage, access and thread control. Based on the type of application it may provide an improvement over say compiler generated codes with the help of directives. 

**How is CUDA different from other GPU programming models like OpenACC and OpenMP?**

CUDA should not be considered an alternative to OpenMP or OpenACC. In fact CUDA complements directive-based programming models and there are defined interoperability strategies between them. You can always start accelerating your code with OpenACC and use CUDA to optimize the most performance critical kernels. For example use OpenACC for data transfer and then pass a device pointer to one of critical CUDA kernels which are written in CUDA. 

# Saving the exercise

If you would like to download this lab for later viewing, it is recommended you go to your browser's file menu (not the Jupyter notebook file menu) and save the complete web page.  This will ensure the images are copied down as well. You can also execute the following cell block to create a zip file of the files you have been working on, and download it with the link below.

In [ ]:
cd ..
rm -f _files.zip
zip -r _files.zip *

**After** executing the above zip command, you should be able to download and save the zip file by holding down Shift key and right-clicking [Here](../_files.zip) then choosing Save Link As.

# Links and Resources
[Introduction to CUDA](https://devblogs.nvidia.com/even-easier-introduction-cuda/)

[NVIDIA Nsight System](https://docs.nvidia.com/nsight-systems/)

[CUDA Toolkit Download](https://developer.nvidia.com/cuda-downloads)

# Licensing 

Copyright © 2022 OpenACC-Standard.org.  This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials may include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.